In [1]:
import requests
import xarray as xr
import csv
import zipfile
import io
import numpy as np
from datetime import datetime, timedelta
import os
import math

In [2]:
synop_stations = {
    "Jósvafő (SYNOP: 12766)": (48.49528, 20.53583),
    "Debrecen (SYNOP: 12882)": (47.48417, 21.60556),
    "Siófok (SYNOP: 12935)": (46.91056, 18.04056)
}


In [3]:
inittime = datetime(2025, 3, 10, 0)
forecast_hours = list(range(0, 361, 6))

In [4]:
current_dir = os.getcwd()
tdk_dir = os.path.join(current_dir)

print(f"Current working directory: {current_dir}")
print(f"TDK directory: {tdk_dir}")

Current working directory: c:\Users\opago\Documents\Projektek\TDK\tdk\!KODOK_TUTI_JOK
TDK directory: c:\Users\opago\Documents\Projektek\TDK\tdk\!KODOK_TUTI_JOK


plsss

In [ ]:
class ECMWF:
    def __init__(self, name, inittime, forecast_hours):
        self.name = name
        self.inittime = inittime
        self.forecast_hours = forecast_hours
        self.dlon, self.dlat, self.lon0, self.lat0 = 0.25, -0.25, -180, 90

    def extractData(self, filename, target_lat, target_lon):
        try:
            i = int(np.round((target_lon - self.lon0) / self.dlon))
            j = int(np.round((target_lat - self.lat0) / self.dlat))
            
            ds_precip = xr.open_dataset(filename, engine="cfgrib", 
                                      backend_kwargs={'filter_by_keys': {'typeOfLevel': 'surface', 'stepType': 'accum'}})
            
            if 'tp' in ds_precip.data_vars:
                precipitation = ds_precip["tp"].values[j, i]
                if self.name == 'ifs':
                    precipitation = precipitation * 1000  # convert to mm
                return precipitation
            return None
            
        except Exception as e:
            print(f"Error extracting from {filename}: {e}")
            return None

    def extract_precipitation_for_station(self, target_lat, target_lon):
        precipitations = []
        save_dir = os.path.join(tdk_dir, f'ecmwf_grib_data_{self.inittime.strftime("%Y%m%d")}', f'{self.name}-0p25')

        # Összesített értékek gyűjtése
        accum_precipitations = []
        for hour in self.forecast_hours:
            filename = f"{self.inittime.strftime('%Y%m%d%H%M%S')}-{hour}h-oper-fc.grib2"
            file_path = os.path.join(save_dir, filename)
            
            if os.path.exists(file_path):
                precip = self.extractData(file_path, target_lat, target_lon)
                accum_precipitations.append(precip if precip is not None else 0.0)
            else:
                accum_precipitations.append(0.0)

        # 6 órás csapadék számítása
        for i in range(len(accum_precipitations)):
            if i == 0:
                # Első időpont: 0-6 óra közötti csapadék
                precipitations.append(accum_precipitations[0])
            else:
                # Következő időpontok: különbség az előző összesített értékből
                precipitations.append(accum_precipitations[i] - accum_precipitations[i-1])

        return precipitations

class AIFS(ECMWF):
    def __init__(self, inittime):
        super().__init__("aifs-single", inittime, forecast_hours)

class IFS(ECMWF):
    def __init__(self, inittime):
        super().__init__("ifs", inittime, forecast_hours)

class MeasuredData:
    def __init__(self, target_lat, target_lon):
        self.target_lat = target_lat
        self.target_lon = target_lon
        self.metadata = []
        self.station_code = None
        self.timestamps = []
        self.precipitation = []

    def load_metadata(self):
        filename = os.path.join(tdk_dir, "stations_meta_auto.csv")
        if not os.path.exists(filename):
            self.download_metadata()
            
        with open(filename, mode="r", encoding="utf-8") as file:
            csv_metadata = csv.reader(file, delimiter=";")
            next(csv_metadata)
            
            for row in csv_metadata:
                if len(row) >= 7:
                    try:
                        lat = float(row[3])
                        lon = float(row[4])
                        station_code = row[0]
                        station_name = row[6]
                        self.metadata.append((lat, lon, station_code, station_name))
                    except (ValueError, IndexError):
                        continue

    def download_metadata(self):
        metadata_url = 'https://odp.met.hu/climate/observations_hungary/hourly/station_meta_auto.csv'
        output_filename = os.path.join(tdk_dir, "stations_meta_auto.csv")
        response = requests.get(metadata_url)
        if response.status_code == 200:
            with open(output_filename, "wb") as file:
                file.write(response.content)

    def find_nearest_station(self):
        if not self.metadata:
            self.load_metadata()
            
        distances = []
        for lat, lon, station_code, station_name in self.metadata:
            distance = np.sqrt((lat - self.target_lat)**2 + (lon - self.target_lon)**2)
            distances.append((distance, station_code, station_name))
                
        nearest_station = min(distances, key=lambda x: x[0])
        return nearest_station[1].strip(), nearest_station[2]

    def download_and_extract_zip(self, station_code):
        station_data_dir = os.path.join(tdk_dir, "station_data")
        if not os.path.exists(station_data_dir):
            os.makedirs(station_data_dir)

        zip_url = f"https://odp.met.hu/climate/observations_hungary/hourly/recent/HABP_1H_{station_code}_akt.zip"
        response = requests.get(zip_url)
        
        if response.status_code == 200:
            with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
                csv_file = zip_ref.namelist()[0]
                zip_ref.extract(csv_file, station_data_dir)
                return os.path.join(station_data_dir, csv_file)
        return None

    def process_csv(self, station_code):
        file_path = self.download_and_extract_zip(station_code)
        
        if not file_path:
            return None

        start_date = inittime
        target_times = [start_date + timedelta(hours=h) for h in forecast_hours]
        
        try:
            with open(file_path, mode="r", encoding="utf-8") as file:
                lines = file.readlines()
                
            data_start = 0
            for i, line in enumerate(lines):
                if line.startswith("StationNumber;") and "Time" in line:
                    data_start = i
                    break
            
            hourly_data = []
            for line in lines[data_start + 1:]:
                if line.strip() == "EOR" or not line.strip():
                    continue
                    
                parts = line.split(';')
                if len(parts) < 3:
                    continue
                    
                try:
                    timestamp_str = parts[1].strip()
                    if len(timestamp_str) >= 12:
                        timestamp = datetime.strptime(timestamp_str, "%Y%m%d%H%M")
                        
                        if timestamp < start_date or timestamp > start_date + timedelta(hours=forecast_hours[-1] + 6):
                            continue
                            
                        precip_str = parts[2].strip()
                        if precip_str and precip_str != '-999':
                            precipitation = float(precip_str)
                        else:
                            precipitation = 0.0
                            
                        hourly_data.append({
                            "timestamp": timestamp,
                            "precipitation": precipitation
                        })
                except (ValueError, IndexError):
                    continue

            precip_6h = [0.0] * len(target_times)
            
            for data in hourly_data:
                timestamp = data["timestamp"]
                precipitation = data["precipitation"]
                
                for i, target_time in enumerate(target_times):
                    period_start = target_time - timedelta(hours=6)
                    if period_start < timestamp <= target_time:
                        precip_6h[i] += precipitation
                        break

            return precip_6h
                
        except Exception as e:
            print(f"Error processing CSV: {e}")
            return None

def calculate_confusion_metrics(measured, predicted, threshold=0.1):
    TP = TN = FP = FN = 0

    valid_pairs = 0
    for obs, pred in zip(measured, predicted):
        if obs is not None and pred is not None:
            valid_pairs += 1
            obs_event = obs >= threshold
            pred_event = pred >= threshold

            if obs_event and pred_event:
                TP += 1
            elif obs_event and not pred_event:
                FN += 1
            elif not obs_event and pred_event:
                FP += 1
            elif not obs_event and not pred_event:
                TN += 1

    print(f"Valid pairs: {valid_pairs}")
    print(f"TP: {TP}, FN: {FN}, FP: {FP}, TN: {TN}")

    bias_score = (TP + FP) / (TP + FN) if (TP + FN) else 0
    pod = TP / (TP + FN) if (TP + FN) else 0
    far = FP / (FP + TP) if (FP + TP) else 0

    return {
        'TP (Hit)': TP,
        'FN (Miss)': FN,
        'FP (False Alarm)': FP,
        'TN (Correct Negative)': TN,
        'Bias score': bias_score,
        'Probability of Detection (POD)': pod,
        'False Alarm Ratio (FAR)': far
    }

def print_confusion_matrix(confusion_metrics, model_name):
    print(f"\n{'='*60}")
    print(f"CONFUSION MATRIX - AGGREGATED - {model_name}")
    print(f"{'='*60}")
    
    metrics = confusion_metrics[model_name]
    
    print("\nConfusion Matrix:")
    print(f"                Predicted")
    print(f"                Yes     No")
    print(f"Actual Yes  |   {metrics['TP (Hit)']:4d}   {metrics['FN (Miss)']:4d}")
    print(f"       No   |   {metrics['FP (False Alarm)']:4d}   {metrics['TN (Correct Negative)']:4d}")
    
    print(f"\nKey Metrics:")
    print(f"Bias Score (BS): {metrics['Bias score']:.3f}")
    print(f"Probability of Detection (POD): {metrics['Probability of Detection (POD)']:.3f}")
    print(f"False Alarm Ratio (FAR): {metrics['False Alarm Ratio (FAR)']:.3f}")
    print(f"{'='*60}")

if __name__ == "__main__":
    aifs_model = AIFS(inittime)
    ifs_model = IFS(inittime)

    common_measured_data = MeasuredData(0, 0)
    common_measured_data.load_metadata()

    all_measured_data = []
    all_AIFS_data = []
    all_IFS_data = []
    
    print(f"Processing {len(synop_stations)} stations...")
    
    for city_name, coords in synop_stations.items():
        target_lat, target_lon = coords
        print(f"\nProcessing: {city_name}")

        # Mért adatok
        measured_data_for_station = MeasuredData(target_lat, target_lon)
        measured_data_for_station.metadata = common_measured_data.metadata
        nearest_station_code, nearest_station_name = measured_data_for_station.find_nearest_station()
        measured_precip = measured_data_for_station.process_csv(nearest_station_code)
        
        if measured_precip is not None:
            all_measured_data.extend(measured_precip)
            print(f"Measured data: {len(measured_precip)} time points")
        else:
            all_measured_data.extend([0.0] * len(forecast_hours))
            print(f"No measured data for {city_name}")

        # AIFS adatok
        aifs_precip = aifs_model.extract_precipitation_for_station(target_lat, target_lon)
        if aifs_precip is not None:
            all_AIFS_data.extend(aifs_precip)
            print(f"AIFS data: {len(aifs_precip)} time points")
        else:
            all_AIFS_data.extend([0.0] * len(forecast_hours))
            print(f"No AIFS data for {city_name}")

        # IFS adatok
        ifs_precip = ifs_model.extract_precipitation_for_station(target_lat, target_lon)
        if ifs_precip is not None:
            all_IFS_data.extend(ifs_precip)
            print(f"IFS data: {len(ifs_precip)} time points")
        else:
            all_IFS_data.extend([0.0] * len(forecast_hours))
            print(f"No IFS data for {city_name}")

    # Eredmények
    print(f"\nTotal measured data points: {len(all_measured_data)}")
    print(f"Total AIFS data points: {len(all_AIFS_data)}")
    print(f"Total IFS data points: {len(all_IFS_data)}")

    confusion_metrics = {
        'AIFS': calculate_confusion_metrics(all_measured_data, all_AIFS_data),
        'IFS': calculate_confusion_metrics(all_measured_data, all_IFS_data)
    }
    
    print_confusion_matrix(confusion_metrics, 'AIFS')
    print_confusion_matrix(confusion_metrics, 'IFS')

Processing 3 stations...

Processing: Jósvafő (SYNOP: 12766)
Measured data: 61 time points
AIFS data: 61 time points
IFS data: 61 time points

Processing: Debrecen (SYNOP: 12882)
Measured data: 61 time points
AIFS data: 61 time points
IFS data: 61 time points

Processing: Siófok (SYNOP: 12935)
Measured data: 61 time points
AIFS data: 61 time points
IFS data: 61 time points

Total measured data points: 183
Total AIFS data points: 183
Total IFS data points: 183
Valid pairs: 183
TP: 0, FN: 53, FP: 0, TN: 130
Valid pairs: 183
TP: 0, FN: 53, FP: 0, TN: 130

CONFUSION MATRIX - AGGREGATED - AIFS

Confusion Matrix:
                Predicted
                Yes     No
Actual Yes  |      0     53
       No   |      0    130

Key Metrics:
Bias Score (BS): 0.000
Probability of Detection (POD): 0.000
False Alarm Ratio (FAR): 0.000

CONFUSION MATRIX - AGGREGATED - IFS

Confusion Matrix:
                Predicted
                Yes     No
Actual Yes  |      0     53
       No   |      0    130

Key 

: 

**IFS gyengecsap-felülbecslése** nem muxik

In [5]:
class ECMWF:
    def __init__(self, name, inittime, forecast_hours):
        self.name = name
        self.inittime = inittime
        self.forecast_hours = forecast_hours
        self.dlon, self.dlat, self.lon0, self.lat0 = 0.25, -0.25, -180, 90

    def extractData(self, filename, target_lat, target_lon):
        try:
            i = int(np.round((target_lon - self.lon0) / self.dlon))
            j = int(np.round((target_lat - self.lat0) / self.dlat))
            
            ds_precip = xr.open_dataset(filename, engine="cfgrib", 
                                      backend_kwargs={'filter_by_keys': {'typeOfLevel': 'surface', 'stepType': 'accum'}})
            
            if 'tp' in ds_precip.data_vars:
                precipitation = ds_precip["tp"].values[j, i]
                if self.name == 'ifs':
                    precipitation = precipitation * 1000  # convert to mm
                return precipitation
            return None
            
        except Exception as e:
            print(f"Error extracting from {filename}: {e}")
            return None

    def extract_precipitation_for_station(self, target_lat, target_lon):
        precipitations = []
        save_dir = os.path.join(tdk_dir, f'ecmwf_grib_data_{self.inittime.strftime("%Y%m%d")}', f'{self.name}-0p25')

        # Összesített értékek gyűjtése
        accum_precipitations = []
        for hour in self.forecast_hours:
            filename = f"{self.inittime.strftime('%Y%m%d%H%M%S')}-{hour}h-oper-fc.grib2"
            file_path = os.path.join(save_dir, filename)
            
            if os.path.exists(file_path):
                precip = self.extractData(file_path, target_lat, target_lon)
                accum_precipitations.append(precip if precip is not None else 0.0)
            else:
                accum_precipitations.append(0.0)

        # 6 órás csapadék számítása
        for i in range(len(accum_precipitations)):
            if i == 0:
                # Első időpont: 0-6 óra közötti csapadék
                precipitations.append(accum_precipitations[0])
            else:
                # Következő időpontok: különbség az előző összesített értékből
                precipitations.append(accum_precipitations[i] - accum_precipitations[i-1])

        return precipitations

class AIFS(ECMWF):
    def __init__(self, inittime):
        super().__init__("aifs-single", inittime, forecast_hours)

class IFS(ECMWF):
    def __init__(self, inittime):
        super().__init__("ifs", inittime, forecast_hours)

class MeasuredData:
    def __init__(self, target_lat, target_lon):
        self.target_lat = target_lat
        self.target_lon = target_lon
        self.metadata = []
        self.station_code = None
        self.timestamps = []
        self.precipitation = []

    def load_metadata(self):
        filename = os.path.join(tdk_dir, "stations_meta_auto.csv")
        if not os.path.exists(filename):
            self.download_metadata()
            
        with open(filename, mode="r", encoding="utf-8") as file:
            csv_metadata = csv.reader(file, delimiter=";")
            next(csv_metadata)
            
            for row in csv_metadata:
                if len(row) >= 7:
                    try:
                        lat = float(row[3])
                        lon = float(row[4])
                        station_code = row[0]
                        station_name = row[6]
                        self.metadata.append((lat, lon, station_code, station_name))
                    except (ValueError, IndexError):
                        continue

    def download_metadata(self):
        metadata_url = 'https://odp.met.hu/climate/observations_hungary/hourly/station_meta_auto.csv'
        output_filename = os.path.join(tdk_dir, "stations_meta_auto.csv")
        response = requests.get(metadata_url)
        if response.status_code == 200:
            with open(output_filename, "wb") as file:
                file.write(response.content)

    def find_nearest_station(self):
        if not self.metadata:
            self.load_metadata()
            
        distances = []
        for lat, lon, station_code, station_name in self.metadata:
            distance = np.sqrt((lat - self.target_lat)**2 + (lon - self.target_lon)**2)
            distances.append((distance, station_code, station_name))
                
        nearest_station = min(distances, key=lambda x: x[0])
        return nearest_station[1].strip(), nearest_station[2]

    def download_and_extract_zip(self, station_code):
        station_data_dir = os.path.join(tdk_dir, "station_data")
        if not os.path.exists(station_data_dir):
            os.makedirs(station_data_dir)

        zip_url = f"https://odp.met.hu/climate/observations_hungary/hourly/recent/HABP_1H_{station_code}_akt.zip"
        response = requests.get(zip_url)
        
        if response.status_code == 200:
            with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
                csv_file = zip_ref.namelist()[0]
                zip_ref.extract(csv_file, station_data_dir)
                return os.path.join(station_data_dir, csv_file)
        return None

    def process_csv(self, station_code):
        file_path = self.download_and_extract_zip(station_code)
        
        if not file_path:
            return None

        start_date = inittime
        target_times = [start_date + timedelta(hours=h) for h in forecast_hours]
        
        try:
            with open(file_path, mode="r", encoding="utf-8") as file:
                lines = file.readlines()
                
            data_start = 0
            for i, line in enumerate(lines):
                if line.startswith("StationNumber;") and "Time" in line:
                    data_start = i
                    break
            
            hourly_data = []
            for line in lines[data_start + 1:]:
                if line.strip() == "EOR" or not line.strip():
                    continue
                    
                parts = line.split(';')
                if len(parts) < 3:
                    continue
                    
                try:
                    timestamp_str = parts[1].strip()
                    if len(timestamp_str) >= 12:
                        timestamp = datetime.strptime(timestamp_str, "%Y%m%d%H%M")
                        
                        if timestamp < start_date or timestamp > start_date + timedelta(hours=forecast_hours[-1] + 6):
                            continue
                            
                        precip_str = parts[2].strip()
                        if precip_str and precip_str != '-999':
                            precipitation = float(precip_str)
                        else:
                            precipitation = 0.0
                            
                        hourly_data.append({
                            "timestamp": timestamp,
                            "precipitation": precipitation
                        })
                except (ValueError, IndexError):
                    continue

            precip_6h = [0.0] * len(target_times)
            
            for data in hourly_data:
                timestamp = data["timestamp"]
                precipitation = data["precipitation"]
                
                for i, target_time in enumerate(target_times):
                    period_start = target_time - timedelta(hours=6)
                    if period_start < timestamp <= target_time:
                        precip_6h[i] += precipitation
                        break

            return precip_6h
                
        except Exception as e:
            print(f"Error processing CSV: {e}")
            return None

In [6]:
def calculate_confusion_metrics(measured, predicted, lower_threshold=0.1, upper_threshold=1.0):
    """Calculate confusion matrix and related metrics for precipitation between 0.1 mm and 1.0 mm."""
    TP = TN = FP = FN = 0

    # Filter out None and NaN values from both lists for consistent comparison
    valid_pairs = []
    for obs, pred in zip(measured, predicted):
        if obs is not None and pred is not None and not np.isnan(obs) and not np.isnan(pred):
            valid_pairs.append((obs, pred))
    
    if not valid_pairs:
        return {
            'TP (Hit)': 0,
            'FN (Miss)': 0,
            'FP (False Alarm)': 0,
            'TN (Correct Negative)': 0,
            'Bias score': np.nan,
            'Probability of Detection (POD)': np.nan,
            'False Alarm Ratio (FAR)': np.nan
        }

    for obs, pred in valid_pairs:
        # Csak 0.1 mm és 1.0 mm közötti csapadékot vesszük figyelembe
        obs_event = lower_threshold <= obs <= upper_threshold
        pred_event = lower_threshold <= pred <= upper_threshold

        if obs_event and pred_event:
            TP += 1
        elif obs_event and not pred_event:
            FN += 1
        elif not obs_event and pred_event:
            FP += 1
        elif not obs_event and not pred_event:
            TN += 1

    # Calculate the requested metrics only
    bias_score = (TP + FP) / (TP + FN) if (TP + FN) else 0
    pod = TP / (TP + FN) if (TP + FN) else 0
    far = FP / (FP + TP) if (FP + TP) else 0

    return {
        'TP (Hit)': TP,
        'FN (Miss)': FN,
        'FP (False Alarm)': FP,
        'TN (Correct Negative)': TN,
        'Bias score': bias_score,
        'Probability of Detection (POD)': pod,
        'False Alarm Ratio (FAR)': far
    }



In [7]:
def print_confusion_matrix(confusion_metrics, model_name):
    print(f"\n{'='*60}")
    print(f"CONFUSION MATRIX - AGGREGATED - {model_name}")
    print(f"{'='*60}")
    
    metrics = confusion_metrics[model_name]
    
    print("\nConfusion Matrix:")
    print(f"                Predicted")
    print(f"                Yes     No")
    print(f"Actual Yes  |   {metrics['TP (Hit)']:4d}   {metrics['FN (Miss)']:4d}")
    print(f"       No   |   {metrics['FP (False Alarm)']:4d}   {metrics['TN (Correct Negative)']:4d}")
    
    print(f"\nKey Metrics:")
    print(f"Bias Score (BS): {metrics['Bias score']:.3f}")
    print(f"Probability of Detection (POD): {metrics['Probability of Detection (POD)']:.3f}")
    print(f"False Alarm Ratio (FAR): {metrics['False Alarm Ratio (FAR)']:.3f}")
    print(f"{'='*60}")

In [8]:
if __name__ == "__main__":
    ifs_model = IFS(inittime)

    common_measured_data = MeasuredData(0, 0)
    common_measured_data.load_metadata()

    all_measured_data = []
    all_IFS_data = []
    
    print(f"Processing {len(synop_stations)} stations...")
    
    for city_name, coords in synop_stations.items():
        target_lat, target_lon = coords
        print(f"\nProcessing: {city_name}")

        # Mért adatok
        measured_data_for_station = MeasuredData(target_lat, target_lon)
        measured_data_for_station.metadata = common_measured_data.metadata
        nearest_station_code, nearest_station_name = measured_data_for_station.find_nearest_station()
        measured_precip = measured_data_for_station.process_csv(nearest_station_code)
        
        if measured_precip is not None:
            all_measured_data.extend(measured_precip)
            print(f"Measured data: {len(measured_precip)} time points")
        else:
            all_measured_data.extend([0.0] * len(forecast_hours))
            print(f"No measured data for {city_name}")

        # IFS adatok
        ifs_precip = ifs_model.extract_precipitation_for_station(target_lat, target_lon)
        if ifs_precip is not None:
            all_IFS_data.extend(ifs_precip)
            print(f"IFS data: {len(ifs_precip)} time points")
        else:
            all_IFS_data.extend([0.0] * len(forecast_hours))
            print(f"No IFS data for {city_name}")

    # Eredmények
    print(f"\nTotal measured data points: {len(all_measured_data)}")
    print(f"Total IFS data points: {len(all_IFS_data)}")

    confusion_metrics = {
        'IFS': calculate_confusion_metrics(all_measured_data, all_IFS_data)
    }
    
    print_confusion_matrix(confusion_metrics, 'IFS')

Processing 3 stations...

Processing: Jósvafő (SYNOP: 12766)
Measured data: 61 time points
IFS data: 61 time points

Processing: Debrecen (SYNOP: 12882)
Measured data: 61 time points
IFS data: 61 time points

Processing: Siófok (SYNOP: 12935)
Measured data: 61 time points
IFS data: 61 time points

Total measured data points: 183
Total IFS data points: 183

CONFUSION MATRIX - AGGREGATED - IFS

Confusion Matrix:
                Predicted
                Yes     No
Actual Yes  |      0     23
       No   |      0    160

Key Metrics:
Bias Score (BS): 0.000
Probability of Detection (POD): 0.000
False Alarm Ratio (FAR): 0.000


mukodjel

In [9]:
class ECMWF:
    def __init__(self, name, inittime, forecast_hours):
        self.name = name
        self.inittime = inittime
        self.forecast_hours = forecast_hours
        self.dlon, self.dlat, self.lon0, self.lat0 = 0.25, -0.25, -180, 90

    def extractData(self, filename, target_lat, target_lon):
        try:
            i = int(np.round((target_lon - self.lon0) / self.dlon))
            j = int(np.round((target_lat - self.lat0) / self.dlat))
            
            ds_precip = xr.open_dataset(filename, engine="cfgrib", 
                                      backend_kwargs={'filter_by_keys': {'typeOfLevel': 'surface', 'stepType': 'accum'}})
            
            if 'tp' in ds_precip.data_vars:
                precipitation = ds_precip["tp"].values[j, i]
                if self.name == 'ifs':
                    precipitation = precipitation * 1000  # convert to mm
                return precipitation
            return None
            
        except Exception as e:
            print(f"Error extracting from {filename}: {e}")
            return None

    def extract_precipitation_for_station(self, target_lat, target_lon):
        precipitations = []
        save_dir = os.path.join(tdk_dir, f'ecmwf_grib_data_{self.inittime.strftime("%Y%m%d")}', f'{self.name}-0p25')

        # Összesített értékek gyűjtése
        accum_precipitations = []
        for hour in self.forecast_hours:
            filename = f"{self.inittime.strftime('%Y%m%d%H%M%S')}-{hour}h-oper-fc.grib2"
            file_path = os.path.join(save_dir, filename)
            
            if os.path.exists(file_path):
                precip = self.extractData(file_path, target_lat, target_lon)
                accum_precipitations.append(precip if precip is not None else 0.0)
            else:
                accum_precipitations.append(0.0)

        # 6 órás csapadék számítása
        for i in range(len(accum_precipitations)):
            if i == 0:
                # Első időpont: 0-6 óra közötti csapadék
                precipitations.append(accum_precipitations[0])
            else:
                # Következő időpontok: különbség az előző összesített értékből
                precipitations.append(accum_precipitations[i] - accum_precipitations[i-1])

        return precipitations

class AIFS(ECMWF):
    def __init__(self, inittime):
        super().__init__("aifs-single", inittime, forecast_hours)

class IFS(ECMWF):
    def __init__(self, inittime):
        super().__init__("ifs", inittime, forecast_hours)

class MeasuredData:
    def __init__(self, target_lat, target_lon):
        self.target_lat = target_lat
        self.target_lon = target_lon
        self.metadata = []
        self.station_code = None
        self.timestamps = []
        self.precipitation = []

    def load_metadata(self):
        filename = os.path.join(tdk_dir, "stations_meta_auto.csv")
        if not os.path.exists(filename):
            self.download_metadata()
            
        with open(filename, mode="r", encoding="utf-8") as file:
            csv_metadata = csv.reader(file, delimiter=";")
            next(csv_metadata)
            
            for row in csv_metadata:
                if len(row) >= 7:
                    try:
                        lat = float(row[3])
                        lon = float(row[4])
                        station_code = row[0]
                        station_name = row[6]
                        self.metadata.append((lat, lon, station_code, station_name))
                    except (ValueError, IndexError):
                        continue

    def download_metadata(self):
        metadata_url = 'https://odp.met.hu/climate/observations_hungary/hourly/station_meta_auto.csv'
        output_filename = os.path.join(tdk_dir, "stations_meta_auto.csv")
        response = requests.get(metadata_url)
        if response.status_code == 200:
            with open(output_filename, "wb") as file:
                file.write(response.content)

    def find_nearest_station(self):
        if not self.metadata:
            self.load_metadata()
            
        distances = []
        for lat, lon, station_code, station_name in self.metadata:
            distance = np.sqrt((lat - self.target_lat)**2 + (lon - self.target_lon)**2)
            distances.append((distance, station_code, station_name))
                
        nearest_station = min(distances, key=lambda x: x[0])
        return nearest_station[1].strip(), nearest_station[2]

    def download_and_extract_zip(self, station_code):
        station_data_dir = os.path.join(tdk_dir, "station_data")
        if not os.path.exists(station_data_dir):
            os.makedirs(station_data_dir)

        zip_url = f"https://odp.met.hu/climate/observations_hungary/hourly/recent/HABP_1H_{station_code}_akt.zip"
        response = requests.get(zip_url)
        
        if response.status_code == 200:
            with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
                csv_file = zip_ref.namelist()[0]
                zip_ref.extract(csv_file, station_data_dir)
                return os.path.join(station_data_dir, csv_file)
        return None

    def process_csv(self, station_code):
        file_path = self.download_and_extract_zip(station_code)
        
        if not file_path:
            return None

        start_date = inittime
        target_times = [start_date + timedelta(hours=h) for h in forecast_hours]
        
        try:
            with open(file_path, mode="r", encoding="utf-8") as file:
                lines = file.readlines()
                
            data_start = 0
            for i, line in enumerate(lines):
                if line.startswith("StationNumber;") and "Time" in line:
                    data_start = i
                    break
            
            hourly_data = []
            for line in lines[data_start + 1:]:
                if line.strip() == "EOR" or not line.strip():
                    continue
                    
                parts = line.split(';')
                if len(parts) < 3:
                    continue
                    
                try:
                    timestamp_str = parts[1].strip()
                    if len(timestamp_str) >= 12:
                        timestamp = datetime.strptime(timestamp_str, "%Y%m%d%H%M")
                        
                        if timestamp < start_date or timestamp > start_date + timedelta(hours=forecast_hours[-1] + 6):
                            continue
                            
                        precip_str = parts[2].strip()
                        if precip_str and precip_str != '-999':
                            precipitation = float(precip_str)
                        else:
                            precipitation = 0.0
                            
                        hourly_data.append({
                            "timestamp": timestamp,
                            "precipitation": precipitation
                        })
                except (ValueError, IndexError):
                    continue

            precip_6h = [0.0] * len(target_times)
            
            for data in hourly_data:
                timestamp = data["timestamp"]
                precipitation = data["precipitation"]
                
                for i, target_time in enumerate(target_times):
                    period_start = target_time - timedelta(hours=6)
                    if period_start < timestamp <= target_time:
                        precip_6h[i] += precipitation
                        break

            return precip_6h
                
        except Exception as e:
            print(f"Error processing CSV: {e}")
            return None

def calculate_confusion_metrics(measured, predicted, lower_threshold=0.1, upper_threshold=1.0):
    """Calculate confusion matrix and related metrics for precipitation between 0.1 mm and 1.0 mm."""
    TP = TN = FP = FN = 0

    # Filter out None and NaN values from both lists for consistent comparison
    valid_pairs = []
    for obs, pred in zip(measured, predicted):
        if obs is not None and pred is not None and not np.isnan(obs) and not np.isnan(pred):
            valid_pairs.append((obs, pred))
    
    if not valid_pairs:
        return {
            'TP (Hit)': 0,
            'FN (Miss)': 0,
            'FP (False Alarm)': 0,
            'TN (Correct Negative)': 0,
            'Bias score': np.nan,
            'Probability of Detection (POD)': np.nan,
            'False Alarm Ratio (FAR)': np.nan
        }

    for obs, pred in valid_pairs:
        # Csak 0.1 mm és 1.0 mm közötti csapadékot vesszük figyelembe
        obs_event = lower_threshold <= obs <= upper_threshold
        pred_event = lower_threshold <= pred <= upper_threshold

        if obs_event and pred_event:
            TP += 1
        elif obs_event and not pred_event:
            FN += 1
        elif not obs_event and pred_event:
            FP += 1
        elif not obs_event and not pred_event:
            TN += 1

    # Calculate the requested metrics only
    bias_score = (TP + FP) / (TP + FN) if (TP + FN) else 0
    pod = TP / (TP + FN) if (TP + FN) else 0
    far = FP / (FP + TP) if (FP + TP) else 0

    return {
        'TP (Hit)': TP,
        'FN (Miss)': FN,
        'FP (False Alarm)': FP,
        'TN (Correct Negative)': TN,
        'Bias score': bias_score,
        'Probability of Detection (POD)': pod,
        'False Alarm Ratio (FAR)': far
    }


def print_confusion_matrix(confusion_metrics, model_name):
    print(f"\n{'='*60}")
    print(f"CONFUSION MATRIX - AGGREGATED - {model_name}")
    print(f"{'='*60}")
    
    metrics = confusion_metrics[model_name]
    
    print("\nConfusion Matrix:")
    print(f"                Predicted")
    print(f"                Yes     No")
    print(f"Actual Yes  |   {metrics['TP (Hit)']:4d}   {metrics['FN (Miss)']:4d}")
    print(f"       No   |   {metrics['FP (False Alarm)']:4d}   {metrics['TN (Correct Negative)']:4d}")
    
    print(f"\nKey Metrics:")
    print(f"Bias Score (BS): {metrics['Bias score']:.3f}")
    print(f"Probability of Detection (POD): {metrics['Probability of Detection (POD)']:.3f}")
    print(f"False Alarm Ratio (FAR): {metrics['False Alarm Ratio (FAR)']:.3f}")
    print(f"{'='*60}")

if __name__ == "__main__":
    aifs_model = AIFS(inittime)
    ifs_model = IFS(inittime)

    common_measured_data = MeasuredData(0, 0)
    common_measured_data.load_metadata()

    all_measured_data = []
    all_AIFS_data = []
    all_IFS_data = []
    
    print(f"Processing {len(synop_stations)} stations...")
    
    for city_name, coords in synop_stations.items():
        target_lat, target_lon = coords
        print(f"\nProcessing: {city_name}")

        # Mért adatok
        measured_data_for_station = MeasuredData(target_lat, target_lon)
        measured_data_for_station.metadata = common_measured_data.metadata
        nearest_station_code, nearest_station_name = measured_data_for_station.find_nearest_station()
        measured_precip = measured_data_for_station.process_csv(nearest_station_code)
        
        if measured_precip is not None:
            all_measured_data.extend(measured_precip)
            print(f"Measured data: {len(measured_precip)} time points")
        else:
            all_measured_data.extend([0.0] * len(forecast_hours))
            print(f"No measured data for {city_name}")

        # AIFS adatok
        aifs_precip = aifs_model.extract_precipitation_for_station(target_lat, target_lon)
        if aifs_precip is not None:
            all_AIFS_data.extend(aifs_precip)
            print(f"AIFS data: {len(aifs_precip)} time points")
        else:
            all_AIFS_data.extend([0.0] * len(forecast_hours))
            print(f"No AIFS data for {city_name}")

        # IFS adatok
        ifs_precip = ifs_model.extract_precipitation_for_station(target_lat, target_lon)
        if ifs_precip is not None:
            all_IFS_data.extend(ifs_precip)
            print(f"IFS data: {len(ifs_precip)} time points")
        else:
            all_IFS_data.extend([0.0] * len(forecast_hours))
            print(f"No IFS data for {city_name}")

    # Eredmények
    print(f"\nTotal measured data points: {len(all_measured_data)}")
    print(f"Total AIFS data points: {len(all_AIFS_data)}")
    print(f"Total IFS data points: {len(all_IFS_data)}")

    confusion_metrics = {
        'AIFS': calculate_confusion_metrics(all_measured_data, all_AIFS_data),
        'IFS': calculate_confusion_metrics(all_measured_data, all_IFS_data)
    }
    
    print_confusion_matrix(confusion_metrics, 'AIFS')
    print_confusion_matrix(confusion_metrics, 'IFS')

Processing 3 stations...

Processing: Jósvafő (SYNOP: 12766)
Measured data: 61 time points
AIFS data: 61 time points
IFS data: 61 time points

Processing: Debrecen (SYNOP: 12882)
Measured data: 61 time points
AIFS data: 61 time points
IFS data: 61 time points

Processing: Siófok (SYNOP: 12935)
Measured data: 61 time points
AIFS data: 61 time points
IFS data: 61 time points

Total measured data points: 183
Total AIFS data points: 183
Total IFS data points: 183

CONFUSION MATRIX - AGGREGATED - AIFS

Confusion Matrix:
                Predicted
                Yes     No
Actual Yes  |      0     23
       No   |      0    160

Key Metrics:
Bias Score (BS): 0.000
Probability of Detection (POD): 0.000
False Alarm Ratio (FAR): 0.000

CONFUSION MATRIX - AGGREGATED - IFS

Confusion Matrix:
                Predicted
                Yes     No
Actual Yes  |      0     23
       No   |      0    160

Key Metrics:
Bias Score (BS): 0.000
Probability of Detection (POD): 0.000
False Alarm Ratio (FAR):